In [ ]:
# 1. Install Needed Packages
!pip install gradio ipywidgets wandb transformers datasets tokenizers accelerate torch matplotlib numpy torchinfo peft -q
!pip install -U datasets

In [ ]:
import sys
print(sys.version)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Imports and Setup
import string
import torch
import numpy as np
import matplotlib.pyplot as plt
from accelerate import Accelerator
import gc
import wandb

from datasets import load_dataset, DatasetDict
from transformers import (
    TrainingArguments,
    Trainer,
    TrainerCallback,
    TrainerState,
    TrainerControl
)
from dataclasses import dataclass

input_size = 256

In [ ]:
import torch
print(f"Number of GPUs: {torch.cuda.device_count()}")
print(f"Current device: {torch.cuda.current_device() if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Initialize Accelerator
accelerator = Accelerator()

# Move model and data to the right device (CPU or GPU)
device = accelerator.device
print(f"Using device: {device}")

In [ ]:
from transformers import AutoTokenizer

base_model = 'Llama-3.2-3B-Instruct'
# Load the tokenizer used by the sentence embedding model

tokenizer = AutoTokenizer.from_pretrained("meta-llama/" + base_model, use_fast=True, token=os.environ.get("HF_TOKEN"))

print("Special Tokens:", tokenizer.special_tokens_map)
print(tokenizer.decode(range(128000,128256)))

In [ ]:
CACHE_ROOT = "/content/drive/MyDrive/cached_datasets/noised_final"
TRAIN_PATH = f"{CACHE_ROOT}/train"
VAL_PATH = f"{CACHE_ROOT}/val"
TEST_PATH = f"{CACHE_ROOT}/test"

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
from tqdm import tqdm
import random
import os

# === Load tokenizer ===
from transformers import AutoTokenizer
tokenizer.pad_token = tokenizer.eos_token
pad_token = tokenizer.pad_token_id
max_len = input_size

if not all(os.path.exists(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]):

    opc_sft = load_dataset("OpenCoder-LLM/opc-sft-stage2", "educational_instruct", split="train")
    mbpp_rl = load_dataset("ankner/mbpp-rl-llama3-8b-base-labeled", split="train")
    orca_math = load_dataset("microsoft/orca-math-word-problems-200k", split="train")

    # === Load and filter datasets ===
    alpaca_dataset = load_dataset("tatsu-lab/alpaca", split="train")
    gpt4_dataset = load_dataset("vicgalle/alpaca-gpt4", split="train")

    streaming_clean = load_dataset("crumb/Clean-Instruct-3M", split="train", streaming=True)
    # filtered_stream = streaming_clean.filter(
    #     lambda x: len(x["output"]) >= 128 and (len(x["input"]) + len(x["output"])) <= 896
    # )
    filtered_stream = streaming_clean.filter(
        lambda x: (len(x["input"]) + len(x["output"])) <= 896
    )
    sampled_clean = list(filtered_stream.shuffle().take(500_000))
    clean_dataset = Dataset.from_list(sampled_clean)

    # === Load ARC and MMLU and HellaSwag and GSM8K ===
    arc_dataset = load_dataset("ai2_arc", "ARC-Easy")["train"]
    mmlu_dataset = load_dataset("cais/mmlu", "all")["auxiliary_train"].shuffle(seed=42)
    hellaswag_dataset = load_dataset("hellaswag", split="train", trust_remote_code=True)
    gsm8k_dataset = load_dataset("openai/gsm8k", "main", split="train")


In [ ]:
def format_mc_example(prompt, target):
    return {
        "instruction": "",
        "input": prompt,
        "output": target,
    }

def format_gsm8k_example(example):
    return {
        "instruction": "Solve the following math problem step by step.",
        "input": example["question"].strip(),
        "output": example["answer"].strip()
    }

def format_hellaswag_example(example):
    prompt = f"{example['ctx'].strip()} Which one of the following options completes the sentence most plausibly?"
    options = example["endings"]
    options_text = "\n".join([f"{chr(65 + i)}: {opt}" for i, opt in enumerate(options)])
    full_prompt = f"{prompt}\n{options_text}"

    try:
        label = int(example["label"])  # Convert label to int
    except:
        return None

    if label >= len(options):
        return None

    target = f"{chr(65 + label)}: {options[label]}"

    if len(full_prompt+target) > (max_len * 3.5):
        return None
    return format_mc_example(full_prompt, target)

def format_mmlu_example(example):
    question = example["question"].strip()
    choices = example["choices"]
    gold_idx = example["answer"]
    if gold_idx is None or gold_idx >= len(choices):
        return None
    labels = list("ABCD")[:len(choices)]
    gold_label = labels[gold_idx]
    answer_text = choices[gold_idx]
    options = "\n".join([f"{l}: {c}" for l, c in zip(labels, choices)])
    prompt = f"""{question}
Which one of the following options is the best answer to this question?:
{options}"""
    target = f"{gold_label}: {answer_text}"
    if len(prompt+target) > (max_len * 3.5):
        return None
    return format_mc_example(prompt, target)

def format_arc_example(example):
    q = example["question"]
    options = example["choices"]["text"]
    labels = example["choices"]["label"]
    gold = example["answerKey"]
    if gold not in labels:
        return None
    idx = labels.index(gold)
    answer_text = options[idx]
    option_str = "\n".join([f"{l}: {t}" for l, t in zip(labels, options)])
    prompt = f"""{q.strip()}
Which one of the following is options is the best answer to this question?:
{option_str}"""
    target = f"{labels[idx]}: {answer_text}"
    if len(prompt+target) > (max_len * 3.5):
        return None
    return format_mc_example(prompt, target)


if not all(os.path.exists(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]):

    orca_math_data = [
        {"instruction": "Solve the following math problem step by step.", "input": ex["question"], "output": ex["answer"]}
        for ex in orca_math if ex["question"] and ex["answer"]
    ]

    mbpp_rl_data = [
        {"instruction": "Write a function to solve the task below.", "input": ex["input"], "output": ex["response"]}
        for ex in mbpp_rl if ex["input"] and ex["response"]
    ]

    opc_sft_data = [
        {"instruction": ex["instruction"], "input": "", "output": ex["output"]}
        for ex in opc_sft if ex["instruction"] and ex["output"]
    ]

    # Format and combine ARC + MMLU
    arc_data = [x for x in (format_arc_example(ex) for ex in arc_dataset) if x is not None]
    mmlu_data = [x for x in (format_mmlu_example(ex) for ex in mmlu_dataset) if x is not None]
    gsm8k_data = [format_gsm8k_example(ex) for ex in gsm8k_dataset]
    hellaswag_data = [x for x in (format_hellaswag_example(ex) for ex in hellaswag_dataset) if x is not None]


    mcqa_dataset = Dataset.from_list(arc_data + mmlu_data)
    hellaswag_formatted = Dataset.from_list(hellaswag_data)
    gsm8k_formatted = Dataset.from_list(gsm8k_data)

In [ ]:
# === Tokenization ===
def format_static(batch):
    input_ids_list = []
    labels_list = []

    for instruction, input_text, response in zip(batch["instruction"], batch["input"], batch["output"]):
        full_instruction = instruction.strip()
        if input_text.strip():
            full_instruction += " " + input_text.strip()

        prompt = (
            "<|begin_of_text|>\n"
            "<|start_header_id|>system<|end_header_id|>\n"
            "You are a helpful assistant.\n"
            "<|eot_id|>\n"
            "<|start_header_id|>user<|end_header_id|>\n"
            f"{full_instruction}\n"
            "<|eot_id|>\n"
            "<|start_header_id|>assistant<|end_header_id|>\n"
            f"{response.strip()}"
        )

        tokenized = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        if len(tokenized) > max_len:
            tokenized = tokenized[:max_len]
        elif len(tokenized) < max_len:
            tokenized += [pad_token] * (max_len - len(tokenized))

        input_ids_list.append(tokenized)
        labels_list.append(tokenized)

    return {"input_ids": input_ids_list, "labels": labels_list}


In [ ]:
if not all(os.path.exists(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]):
    # Tokenize all
    clean_dataset = clean_dataset.shuffle(seed=42).select(range(500_000))

    formatted_alpaca = alpaca_dataset.map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing Alpaca")
    formatted_gpt4 = gpt4_dataset.map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing GPT4")
    formatted_clean = clean_dataset.map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing Clean")
    formatted_mcqa = mcqa_dataset.map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing ARC+MMLU")
    formatted_hellaswag = hellaswag_formatted.map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing Hellaswag")
    formatted_gsm8k = gsm8k_formatted.map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing GSM8K")
    opc_sft_formatted = Dataset.from_list(opc_sft_data).map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing OpenCoder")
    mbpp_rl_formatted = Dataset.from_list(mbpp_rl_data).map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing MBPP-RL")
    orca_math_formatted = Dataset.from_list(orca_math_data).map(format_static, batched=True, batch_size=256, num_proc=8, desc="Tokenizing OrcaMath")


    # Final concat
    formatted_concatenated = concatenate_datasets([
        formatted_alpaca,
        formatted_gpt4,
        formatted_clean,
        formatted_mcqa,
        formatted_hellaswag,
        formatted_gsm8k,
        orca_math_formatted,
        mbpp_rl_formatted,
        opc_sft_formatted
    ]).shuffle(seed=42)


    # Optional cleanup
    del formatted_clean
    del formatted_alpaca
    del formatted_mcqa
    del formatted_hellaswag
    del formatted_gsm8k
    del formatted_gpt4

In [ ]:
if not all(os.path.exists(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]):
    split_data = formatted_concatenated.train_test_split(test_size=0.02, seed=42)
    val_test_split = split_data["test"].train_test_split(test_size=0.5, seed=42)

    raw_datasets = DatasetDict({
        "train": split_data["train"],
        "validation": val_test_split["train"],
        "test": val_test_split["test"]
    })

In [ ]:
import numpy as np
import random

vocab_size = len(tokenizer)
rng = np.random.default_rng()

# Precompute filtered token distribution excluding the EOT token (128001)
eot_token_id = 128001

assistant_marker_ids = tokenizer.encode("<|start_header_id|>assistant<|end_header_id|>", add_special_tokens=False)

mask_token_id = tokenizer.encode('MASK', add_special_tokens=False)[0]


def structurally_corrupt(tokens, noise_prob=0.5):
    tokens = np.array(tokens, dtype=np.int32)
    corrupted = tokens.copy()
    length = len(tokens)

    # 1. Apply random masking (50% chance for this entire block)
    if rng.random() < 0.5:
        mask_fraction = rng.uniform(0.0, 0.5)  # Determine % of tokens to mask (0% to 50%)
        num_to_mask = int(length * mask_fraction)
        if num_to_mask > 0:
            indices_to_mask = rng.choice(
                np.arange(length),
                size=num_to_mask,
                replace=False
            )
            corrupted[indices_to_mask] = mask_token_id

    if length > 2:
        swap_mask = rng.random(length - 1) < (noise_prob / 4)
        dup_mask = rng.random(length) < (noise_prob / 4)

        # Swaps
        for i in np.where(swap_mask)[0]:
            corrupted[i], corrupted[i + 1] = corrupted[i + 1], corrupted[i]

        # Duplicates (optional — 50/50 backward/forward copy)
        dup_indices = np.where(dup_mask)[0]
        if len(dup_indices) > 0:
            direction = rng.integers(0, 2, size=len(dup_indices))  # 0 = backward, 1 = forward

            # Ensure we stay within bounds
            backward = dup_indices[(direction == 0) & (dup_indices > 0)]
            forward = dup_indices[(direction == 1) & (dup_indices < len(corrupted) - 1)]

            # Apply duplication
            corrupted[backward] = corrupted[backward - 1]
            corrupted[forward] = corrupted[forward + 1]

        # Random span shift (only if sequence is long enough)
        max_span = min(3, 10)
        if rng.random() < (noise_prob / 4):
            span_len = rng.integers(1, max_span + 1)
            if length - span_len + 1 > 0:
                shift = rng.integers(1, 5)
                direction = rng.choice([-1, 1])
                start = rng.integers(0, length - span_len + 1)
                end = start + span_len
                span = corrupted[start:end]

                if direction == -1:
                    target = max(0, start - shift)
                else:
                    target = min(length - span_len, start + shift)

                insert_len = min(span_len, length - target)
                corrupted[target:target + insert_len] = span[:insert_len]

    return corrupted


def noise_answer_tokens(batch):
    noised_inputs = []
    labels = []

    input_id_list = batch["input_ids"]
    label_id_list = batch["labels"]

    for i, (input_ids, label_ids) in enumerate(zip(input_id_list, label_id_list)):
        input_ids = np.array(input_ids, dtype=np.int32)
        label_ids = np.array(label_ids, dtype=np.int32)

        # Find "Assistant:" marker
        start_idx = -1
        for j in range(len(input_ids) - len(assistant_marker_ids) + 1):
            if np.array_equal(input_ids[j:j + len(assistant_marker_ids)], assistant_marker_ids):
                start_idx = j + len(assistant_marker_ids) - 1
                break
        if start_idx == -1:
            start_idx = 255  # fallback

        instr_part = input_ids[:start_idx + 1]
        answer_part = input_ids[start_idx + 1:]

        # With 1/10 probability, replace the entire answer_part with MASK tokens
        if rng.random() < 0.1:
            if len(answer_part) > 0:
                # Create an array of MASK tokens with the same shape and type as answer_part
                corrupted_answer = np.full_like(answer_part, fill_value=mask_token_id, dtype=np.int32)
            else:
                corrupted_answer = answer_part.copy() # No change if answer_part is empty
        else:
            # Regular structural noise
            noise_prob = rng.uniform(0.0, 1.0)
            corrupted_answer = structurally_corrupt(answer_part, noise_prob=noise_prob)

        noised_input = np.concatenate([instr_part, corrupted_answer[:len(answer_part)]])
        noised_inputs.append(noised_input.tolist())
        labels.append(label_ids.tolist())

    return {
        "input_ids": noised_inputs,
        "labels": labels
    }



# Process datasets
if not all(os.path.exists(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]):

    raw_datasets_subset = raw_datasets["train"].shuffle(seed=42).select(range(50000))

    train_dataset = raw_datasets["train"].map(
        noise_answer_tokens,
        batched=True,
        batch_size=256,
        num_proc=8,
        load_from_cache_file=False,
        desc="Noising training data"
    ).shuffle(seed=41)

    val_dataset = raw_datasets["validation"].map(
        noise_answer_tokens,
        batched=True,
        batch_size=256,
        num_proc=1,
        desc="Noising validation data"
    ).shuffle(seed=41)

    test_dataset = raw_datasets["test"].map(
        noise_answer_tokens,
        batched=True,
        batch_size=256,
        num_proc=1,
        desc="Noising test data"
    ).shuffle(seed=41)

In [ ]:
from datasets import load_from_disk

if all(os.path.exists(p) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]):
    print("Loading cached datasets...")
    train_dataset = load_from_disk(TRAIN_PATH)
    val_dataset = load_from_disk(VAL_PATH)
    test_dataset = load_from_disk(TEST_PATH)
else:
    os.makedirs(CACHE_ROOT, exist_ok=True)
    train_dataset.save_to_disk(TRAIN_PATH)
    val_dataset.save_to_disk(VAL_PATH)
    test_dataset.save_to_disk(TEST_PATH)

In [ ]:
r = random.randint(0, len(train_dataset))
print(tokenizer.decode(train_dataset[r]['labels'], skip_special_tokens=True))
print(tokenizer.decode(train_dataset[r]['input_ids'], skip_special_tokens=True))

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class DiffusionDataCollator:

    def __call__(self, features):
        # Directly stack tensors instead of using a for loop
        batch_input_ids = torch.stack([torch.tensor(f["input_ids"], dtype=torch.long) for f in features])
        batch_labels = torch.stack([torch.tensor(f["labels"], dtype=torch.long) for f in features])

        return {"input_ids": batch_input_ids, "labels": batch_labels}

# Initialize the collator
data_collator = DiffusionDataCollator()

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import torch.nn as nn
from transformers import AutoModelForCausalLM
import torch.nn.init as init
import math

def disable_dropout(model):
    for name, module in model.named_modules():
        if isinstance(module, nn.Dropout):
            setattr(model, name, nn.Identity())  # Replace Dropout with Identity
    return model

# Function to reset all model parameters
def reset_model_weights(model):
    # Loop through all layers and reinitialize weights
    for module in model.modules():
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.reset_parameters()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
        elif isinstance(module, nn.Conv1d):  # If any conv layers exist (Llama) uses them for attention)
            init.kaiming_uniform_(module.weight, a=math.sqrt(5))
            if module.bias is not None:
                fan_in, _ = init._calculate_fan_in_and_fan_out(module.weight)
                bound = 1 / math.sqrt(fan_in)
                init.uniform_(module.bias, -bound, bound)

    return model


In [ ]:
# import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PretrainedConfig
from transformers.models.llama.modeling_llama import LlamaAttention
import matplotlib.pyplot as plt
from torch.amp import autocast
from torch.nn import Embedding
import torch.nn.functional as F
import torchinfo
from peft import LoraConfig, get_peft_model, PeftModel, LoraModel
import importlib
from typing import Callable, List, Optional, Tuple, Union


class CustomTransformerConfig(PretrainedConfig):
    def __init__(self, vocab_size=128256, hidden_size=4096, num_layers=32, num_heads=32, prediction_chunk=256, dropout=0,
                 max_position_embeddings=4096, masking_type="bidirectional", **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout = dropout
        self.prediction_chunk = prediction_chunk
        self.max_position_embeddings = max_position_embeddings
        self.input_size = prediction_chunk
        self.masking_type = masking_type

class CustomTransformerModel(PreTrainedModel):
    # config_class = CustomTransformerConfig

    def __init__(self, config):
        super().__init__(config)
        self.llama = AutoModelForCausalLM.from_pretrained("meta-llama/" + base_model, torch_dtype=torch.float16, device_map="auto", token=os.environ.get("HF_TOKEN"))
        self.llama.resize_token_embeddings(config.vocab_size)

        for param in self.llama.parameters():
            param.requires_grad = False
        for param in self.llama.lm_head.parameters():
            param.requires_grad = True

        lora_config = LoraConfig(
            r=4096, lora_alpha=4096, lora_dropout=0.0,
            target_modules=["q_proj", "v_proj"],
            bias="none", task_type=None
        )

        self.llama = get_peft_model(self.llama, lora_config)
        self.llama.print_trainable_parameters()
        # self.llama = self.llama.to(torch.float16)

    def forward(self, input_ids, labels=None, **kwargs):
        batch_size, seq_len = input_ids.shape
        assert seq_len == self.config.prediction_chunk, f"Expected input length {self.config.prediction_chunk}, got {seq_len}"

        # Build attention mask
        device = input_ids.device

        masking_type = getattr(self.config, "masking_type")
        if masking_type == 'bidirectional':
            base_mask = torch.ones(seq_len, seq_len, dtype=torch.bool, device=device)
        elif masking_type == 'bidirectional_masked':
            base_mask = torch.ones(seq_len, seq_len, dtype=torch.bool, device=device)
            base_mask.fill_diagonal_(False)
        elif masking_type == 'unidirectional':
            base_mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))
        else:
            raise ValueError(f"Unknown masking type: {self.config.masking_type}")

        attention_mask = base_mask.unsqueeze(0).unsqueeze(1).expand(batch_size, 1, seq_len, seq_len).clone()
        attention_mask = attention_mask.to(dtype=torch.float32)  # required for SDPA and Flash attention


        with autocast("cuda", dtype=torch.float16):
            outputs = self.llama(
                input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
                **kwargs
            )

        logits = outputs.logits[:, :, :self.config.vocab_size].view(batch_size, seq_len, self.config.vocab_size)

        loss = None
        if labels is not None:
            assert labels.shape == (batch_size, seq_len), f"Labels shape mismatch: expected ({batch_size}, {seq_len}), got {labels.shape}"
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.config.vocab_size), labels.view(-1))

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

def reset_weights(model):
    for module in model.modules():
        if hasattr(module, "reset_parameters"):  # Check if the layer has reset_parameters()
            module.reset_parameters()
    return model

# Initialize model
new = True
# Initialize Accelerator
accelerator = Accelerator()
# Move model and data to the right device (CPU or GPU)
device = accelerator.device
print(f"Using device: {device}")
# device = "cuda" if torch.cuda.is_available() else "cpu"
if new:
    config = CustomTransformerConfig(vocab_size=len(tokenizer), device=device)
    model = CustomTransformerModel(config)
    model.to(device)
    model = disable_dropout(model)
else:
    checkpoint_path = "./drive/MyDrive/diffusion-model-llama-3.1-8B-finetuned-1024lora-instruct_step20000.pt"
    # Load the model, specifying the PyTorch format and dtype
    config = CustomTransformerConfig(vocab_size=len(tokenizer), device = device)  # Ensure correct vocab size
    # model = CustomTransformerModel(config)
    # model.load_state_dict(torch.load("diffusion_model.pth", weights_only = False))

    ## This is really annoying but necessary to load an old model with only the weights.
    # Get missing classes/functions
    torch.serialization.clear_safe_globals()
    # Step 1: Find missing classes
    unsafe_globals = torch.serialization.get_unsafe_globals_in_checkpoint(checkpoint_path)

    # Step 2: Extract missing class names
    missing_class_names = [name.split(".")[-1] for name in unsafe_globals]

    # Step 3: Try importing from `globals()`, and also attempt dynamic import for Hugging Face classes
    safe_classes = [cls for name, cls in globals().items() if name in missing_class_names]

    for class_path in unsafe_globals:
        try:
            module_name, class_name = class_path.rsplit(".", 1)
            module = importlib.import_module(module_name)
            cls = getattr(module, class_name)
            safe_classes.append(cls)
        except (ImportError, AttributeError) as e:
            print(f"⚠️ Warning: Could not import {class_path} - {e}")

    # Step 4: Register all safe classes
    torch.serialization.add_safe_globals(safe_classes)

    # Step 5: Load the model safely
    model = torch.load(checkpoint_path, weights_only=True)

    # model = model.to(torch.float16) # DO NOT USE THIS, JUST LEFT IT AS A WARNING

    # state_dict = torch.load(checkpoint_path, weights_only=True, map_location=device)
    # model.load_state_dict(state_dict)

    # Disable dropout
    model = disable_dropout(model)

    # Move to GPU if available
    model.to(device)

    model.train()


    print("✅ Model successfully loaded from checkpoint:", checkpoint_path)


# Remove problematic attributes from config
if hasattr(model.config, "device"):
    del model.config.device

print(torchinfo.summary(model,depth = 3))

In [ ]:
torch.cuda.empty_cache()
gc.collect()

# # Freeze llama to retain pre-trained knowledge
# for param in model.llama.parameters():
#     param.requires_grad = True

In [ ]:
!nvidia-smi

In [ ]:
import torch.nn.functional as F
from transformers import TrainerCallback
import wandb

torch.cuda.empty_cache()
train_dataset = train_dataset.shuffle()

class SaveModelCallback(TrainerCallback):
    def __init__(self, save_path_base, save_every=10000):
        self.save_path_base = save_path_base  # e.g., "drive/MyDrive/.../diffusion-model"
        self.save_every = save_every

        # Ensure folder exists at init
        os.makedirs(self.save_path_base, exist_ok=True)

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.save_every == 0 and state.global_step > 0:
            if model is not None:
                save_path = os.path.join(self.save_path_base, f"model_step_{state.global_step}.pt")
                print(f"[Step {state.global_step}] Saving model to {save_path}...")
                torch.save(model, save_path)


# Pass callback to Trainer
specifics = "512LoRA"
save_callback = SaveModelCallback("./drive/MyDrive/Diffusion Models/" + base_model + "_" + specifics)

batch_size = 8

wandb.finish()  # Finish previous run if it exists
wandb.init(project="huggingface", name=f"run-{wandb.util.generate_id()}", reinit=True, allow_val_change=True, mode="online")

# torch.autograd.set_detect_anomaly(True)

# 8. Set Up TrainingArguments
training_args = TrainingArguments(
    num_train_epochs=1,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    save_total_limit=1,  # Keeps only the latest checkpoint
    save_steps=5000,
    logging_steps=100,
    logging_dir="./logs",
    report_to="wandb",
    disable_tqdm=False,
    weight_decay=0.01,
    learning_rate=1e-5,
    warmup_steps=100,
    fp16 = True,
    lr_scheduler_type="cosine",
    remove_unused_columns=False,
    dataloader_num_workers=4,  # Use multiple workers for faster data loading
    max_grad_norm=0.5,  # or 0.5
)

class SaveFixedCheckpointCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is not None and state.global_step % args.save_steps == 0:
            print("Saving model...")
            model.save_pretrained(args.output_dir)

class DiffusionSampleCallback(TrainerCallback):
    def __init__(self, dataset, tokenizer, num_samples=5, token_display_width=10):
        super().__init__()
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.num_samples = num_samples
        self.token_display_width = token_display_width  # Fixed display width per token

    def format_tokens(self, token_ids):
        # Decode tokens using the provided decoding method and split into subwords
        decoded_text = tokenizer.decode(token_ids)
        tokens = decoded_text.split()  # Split into subwords/words

        # Format each token to fit within the fixed width
        formatted_tokens = [
            token[:self.token_display_width].ljust(self.token_display_width)  # Truncate and pad
            for token in tokens
        ]
        return formatted_tokens

    def on_evaluate(self, args, state: TrainerState, control: TrainerControl, trainer=None, format_special = False, **kwargs):
        model.eval()
        device = model.device

        # Sample examples from the dataset
        sampled_indices = [int(idx) for idx in np.random.choice(len(self.dataset), self.num_samples, replace=False)]
        samples = [self.dataset[i] for i in sampled_indices]

        print("\n=== Diffusion Validation Samples ===")
        with torch.no_grad():
            for example in samples:
                input_ids = torch.tensor(example["input_ids"], dtype=torch.long).unsqueeze(0).to(device)
                labels = torch.tensor(example["labels"], dtype=torch.long).unsqueeze(0).to(device)
                outputs = model(input_ids=input_ids)

                logits = outputs['logits']
                # predictions = torch.argmax(logits, dim=-1)
                probabilities = F.softmax(logits / 1.0, dim=-1)  # 🚀 Try temperature scaling
                probabilities = torch.clamp(probabilities, min=1e-8, max=1.0).squeeze()  # 🚀 Prevents NaNs
                predictions = torch.multinomial(probabilities.squeeze(0), num_samples=1).squeeze(-1)

                if format_special:

                    # Format token sequences for display
                    original_tokens = self.format_tokens(labels[0].tolist())
                    input_tokens = self.format_tokens(input_ids[0].tolist())
                    generated_tokens = self.format_tokens(predictions.tolist())

                    # Display original, input, and generated tokens aligned row by row
                    print("\nOriginal  : " + " ".join(original_tokens))
                    print("Input     : " + " ".join(input_tokens))
                    print("Generated : " + " ".join(generated_tokens))

                else:
                    print("\nOriginal  :", tokenizer.decode(labels[0].tolist()))
                    print("Input     :", tokenizer.decode(input_ids[0].tolist()))
                    print("Generated :", tokenizer.decode(predictions.tolist()))


# 10. Reduce Validation Set to 500 Samples
val_subset = val_dataset.select(range(np.min([len(val_dataset),200])))
# val_subset = val_dataset.sample(100)

# Pass the validation or test dataset and tokenizer to the callback
diffusion_callback = DiffusionSampleCallback(
    dataset=test_dataset,  # Or test_dataset
    tokenizer=tokenizer,
    num_samples=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_subset,
    data_collator=data_collator,
    callbacks=[diffusion_callback, save_callback] #, ProgressiveUnfreezingCallback(model)]
)

trainer.train()

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import random

# Setup
vocab_size = len(tokenizer)
rng = np.random.default_rng()

# Token sequence that marks start of assistant's answer
assistant_marker_ids = tokenizer.encode("Assistant:", add_special_tokens=False)
assistant_marker_ids = tokenizer.encode("<|start_header_id|>assistant<|end_header_id|>\n", add_special_tokens=False)

# === Utility: Decode safely ===
def decode_tokens_safe(token_ids):
    return tokenizer.decode(token_ids, skip_special_tokens=True).replace("\n", " ")

# === Utility: Locate assistant start in full input_ids ===
def find_answer_start(input_ids, assistant_marker_ids):
    for i in range(len(input_ids) - len(assistant_marker_ids) + 1):
        if input_ids[i:i+len(assistant_marker_ids)] == assistant_marker_ids:
            return i + len(assistant_marker_ids)
    return None


# === Noising: Only noise the answer part ===
def noisify_answer(input_ids, answer_start, threshold=1.0):
    noised = input_ids.copy()
    answer_len = len(input_ids) - answer_start
    num_to_noise = int(threshold * answer_len)
    if num_to_noise > 0:
        indices = rng.choice(np.arange(answer_start, len(input_ids)), size=num_to_noise, replace=False)
        for idx in indices:
            noised[idx] = tokenizer.encode('MASK', add_special_tokens=False)[0]
    return noised

# === Utility: Define noising schedule ===
def get_noising_schedule(i, max_it, sharpness=5.0):
    x = i / max_it
    return (np.exp(-sharpness * x) - np.exp(-sharpness)) / (1 - np.exp(-sharpness))

# === Diffusion-style one-step refinement ===
def generate_diffusion_text(model, input_ids, answer_start):
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor([input_ids], dtype=torch.long).to(model.device)
        logits = model(input_ids=input_tensor)["logits"]
        probs = F.softmax(logits / 1.0, dim=-1).squeeze()
        probs = torch.clamp(probs, min=1e-8, max=1.0)
        sampled = torch.multinomial(probs, num_samples=1).squeeze().tolist()
    return input_ids[:answer_start] + sampled[answer_start:]

def calculate_answer_perplexity(prompt, answer, model_name='gpt2-large'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    full_input = prompt + answer
    enc = tokenizer(full_input, return_tensors="pt")
    input_ids = enc.input_ids.to(device)

    with torch.no_grad():
        labels = input_ids.clone()
        # Mask prompt part
        prompt_len = len(tokenizer(prompt, add_special_tokens=False)["input_ids"])
        labels[0, :prompt_len] = -100

        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        ppl = torch.exp(loss)
    return ppl.item()

# === Pick random test sample ===
max_it = 128
ground_truth_perplexities = []
generated_perplexities = []

for m in range(1):
    ori_sample = test_dataset[random.randint(0, len(test_dataset) - 1)]
    ori_input_tokens = ori_sample["input_ids"]
    label_tokens = ori_sample["labels"]

    # === Find start of assistant's answer
    answer_start = find_answer_start(ori_input_tokens, assistant_marker_ids)
    if answer_start is None:
        print("❌ No 'Assistant:' token found — skipping")
    else:
        # print("✅ Answer starts at token:", answer_start)

        print("\n--- PROMPT ---")
        print("Question:", decode_tokens_safe(ori_input_tokens[:answer_start]))
        print("Ground Truth:", decode_tokens_safe(label_tokens[answer_start:]))

        # === Initialize with noisy answer
        threshold = 1
        current_tokens = noisify_answer(ori_input_tokens, answer_start, threshold=threshold)
        # print("\nNoisy Start:", decode_tokens_safe(current_tokens[answer_start:]))

        # === Run diffusion loop ===
        history = []
        for i in range(max_it):
            current_tokens = generate_diffusion_text(model, current_tokens, answer_start)
            history.append(current_tokens)

            print(f"[{i:02d}] Noise: [{threshold:.2f}] Generated:", decode_tokens_safe(current_tokens[answer_start:]))

            if len(history) >= 3 and history[-1] == history[-2] == history[-3]:
                print(f"\n✅ Converged at step {i}")
                break

            # Optional: cosine decay of noise
            if i <= max_it:
                threshold = 0.2 * get_noising_schedule(i, max_it, sharpness=5.0)
                current_tokens =  noisify_answer(current_tokens, answer_start, threshold=threshold)

        prompt = decode_tokens_safe(label_tokens[:answer_start])
        ground_truth = decode_tokens_safe(label_tokens[answer_start:])
        generated_answer = decode_tokens_safe(current_tokens[answer_start:])
        ground_truth_perplexity = calculate_answer_perplexity(prompt,ground_truth)
        if ground_truth_perplexity < 1000:
            ground_truth_perplexities.append(ground_truth_perplexity)
        generated_perplexity = calculate_answer_perplexity(prompt,generated_answer)
        if generated_perplexity < 1000:
            generated_perplexities.append(generated_perplexity)
        print("Ground Truth:", ground_truth)
        print("\nGround Truth Perplexity:", ground_truth_perplexity)
        print("\nFinal Answer:", generated_answer)
        print("\nFinal Answer Perplexity:", generated_perplexity)

print("\nAverage Ground Truth Perplexity:", np.mean(ground_truth_perplexities))
print("\nAverage Generated Perplexity:", np.mean(generated_perplexities))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


# Example
answer_start = find_answer_start(label_tokens, assistant_marker_ids)
print(answer_start)
prompt = decode_tokens_safe(label_tokens[:answer_start])
ground_truth = decode_tokens_safe(label_tokens[answer_start:])
generated_answer = decode_tokens_safe(current_tokens[answer_start:])
print("Ground Truth:", ground_truth)
print("\nPerplexity:", calculate_answer_perplexity(prompt,ground_truth))
print("\nFinal Answer:", generated_answer)
print("\nPerplexity:", calculate_answer_perplexity(prompt,generated_answer))


In [ ]:
import torch
import torch.nn.functional as F
import time
import numpy as np
from IPython.display import display, HTML, Markdown, clear_output

def top_k_top_p_filtering(logits, top_k=0, top_p=1.0, filter_value=-float('Inf')):
    if logits.dim() == 1:
        logits = logits.unsqueeze(0)  # Ensure 2D shape [1, vocab_size]

    top_k = min(top_k, logits.size(-1))  # Safety check

    if top_k > 0:
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits = logits.masked_fill(indices_to_remove, filter_value)

    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        # Remove tokens with cumulative prob above threshold
        sorted_mask = cumulative_probs > top_p
        sorted_mask[..., 0] = 0  # Ensure at least one token is kept

        # Now scatter the mask back to original indices
        indices_to_remove = torch.zeros_like(logits, dtype=torch.bool).scatter(1, sorted_indices, sorted_mask)
        logits = logits.masked_fill(indices_to_remove, filter_value)

    return logits.squeeze(0)


def generate_diffusion_text(model, input_ids, answer_start, top_k=1, top_p=0.9, temperature=1.0):
    model.eval()
    with torch.no_grad():
        input_tensor = torch.tensor([input_ids], dtype=torch.long).to(model.device)
        logits = model(input_ids=input_tensor)["logits"].squeeze(0)  # [seq_len, vocab_size]

        # Apply temperature
        logits = logits / temperature

        # Prepare list to hold sampled tokens and confidences
        sampled = []
        confidences = []

        for i in range(logits.size(0)):
            if i < answer_start:
                sampled.append(input_ids[i])
                confidences.append(1.0)
                continue

            logit = logits[i]
            filtered_logits = top_k_top_p_filtering(logit, top_k=top_k, top_p=top_p)
            # filtered_logits = logit
            probs = F.softmax(filtered_logits, dim=-1)
            probs = torch.clamp(probs, min=1e-8, max=1.0)

            token = torch.multinomial(probs, num_samples=1).item()
            confidence = probs[token].item()

            sampled.append(token)
            confidences.append(confidence)

    return sampled, confidences



# === Format token with confidence and color ===
eot_token_id = 128001

def format_token_colored_inline(token_id, conf, tokenizer):
    if token_id == eot_token_id:
        return ""  # skip EOT
    token_str = tokenizer.convert_tokens_to_string([token_id])
    token_str = token_str.replace(" ", "&nbsp;").replace("\n", "<br>")
    color = f"hsl({int(conf * 120)}, 100%, 25%)"
    return f"<span style='color:{color}' title='Conf: {conf:.2f}'>{token_str}</span>"

# === Manual question
question = "Tell me all you know about New York!"
question = "What do you know about Amsterdam?"
# question = "What is the capital of the Colombia? Answer with only one word!"

# question = "What animal has the largest penis? Choose one of the following: A. Giraffe, B. Elephant, C. Monkey, D. Blue whale."

# question = "Is a chicken longer than a snake?"
# prompt = f"User: {question.strip()}\nAssistant:"
prompt = (
    "<|begin_of_text|>\n"
    "<|start_header_id|>system<|end_header_id|>\n"
    "You are a helpful assistant.\n"
    "<|eot_id|>\n"
    "<|start_header_id|>user<|end_header_id|>\n"
    f"{question.strip()}\n"
    "<|eot_id|>\n"
    "<|start_header_id|>assistant<|end_header_id|>\n"
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
assistant_marker_ids = tokenizer.encode("<|start_header_id|>assistant<|end_header_id|>\n", add_special_tokens=False)

assistant_marker_ids = [128006, 78191, 128007, 198]
input_ids = tokenizer.encode(prompt, add_special_tokens=False)
answer_start = find_answer_start(input_ids, assistant_marker_ids)

if answer_start is None:
    raise ValueError("Could not find Assistant: marker in input.")

# Pad to 256
pad_token = tokenizer.pad_token_id or tokenizer.eos_token_id
if len(input_ids) < 256:
    input_ids += [pad_token] * (256 - len(input_ids))
else:
    input_ids = input_ids[:256]

ori_input_tokens = input_ids
baseline_input_ids = torch.tensor([ori_input_tokens[:answer_start]], dtype=torch.long).to(model.device)

# === Start with fully noised answer
current_tokens = noisify_answer(ori_input_tokens, answer_start, threshold=1.0)

# === Diffusion loop
max_it = 64
tokenizer.pad_token_id = tokenizer.eos_token_id
last_tokens = []

# === Format tokens as inline text with color and tooltip ===
def format_token_colored_inline(token_id, conf, tokenizer):
    token_str = tokenizer.decode([token_id]).replace("\n", " ")
    color = f"hsl({int(conf * 120)}, 100%, 25%)"  # red (0) → green (1)
    return f"<span style='color:{color}' title='Conf: {conf:.2f}'>{token_str}</span>"


for i in range(max_it):
    # === Diffusion step ===
    generated_tokens, confidences = generate_diffusion_text(model, current_tokens, answer_start, top_k=100, top_p=1.0, temperature=1.0)
    current_tokens = generated_tokens
    print(tokenizer.decode(generated_tokens))

    # === Display output
    clear_output(wait=True)
    display(Markdown(f"### Iteration {i}/{max_it-1}"))
    display(Markdown(f"**Question:** {decode_tokens_safe(ori_input_tokens[:answer_start])}"))

    # Format output tokens inline with color
    eot_token_id = 128001
    output_html = ''.join([
        format_token_colored_inline(tok, conf, tokenizer)
        for tok, conf in zip(generated_tokens[answer_start:], confidences[answer_start:])
        if tok != eot_token_id
    ])
    display(HTML(f"<b>Diffusion Output with Confidence:</b><br><div style='line-height:1.8'>{output_html}</div>"))

    # === Stop if output hasn't changed for 3 iterations
    last_tokens.append(generated_tokens)
    if len(last_tokens) > 3:
        last_tokens.pop(0)
    if len(last_tokens) == 3 and last_tokens[0] == last_tokens[1] == last_tokens[2]:
        break

    # === Prepare next input for diffusion
    if i < max_it-1:
        threshold = 0.5 * get_noising_schedule(i, max_it, sharpness=5)
        # threshold = 0
        current_tokens = noisify_answer(current_tokens, answer_start, threshold=threshold)

    # time.sleep(0.1)
print("Generated tokens: ",next((i for i, t in enumerate(current_tokens) if t == tokenizer.eos_token_id), len(current_tokens))-answer_start)


In [ ]:
# torch.save(model, "./drive/MyDrive/diffusion-model-llama-3.2-3B-finetuned-1024lora-noninstruct-trainedfurther_step8000.pth")

In [ ]:
# import openai
# import torch.nn.functional as F
# from transformers import TrainerCallback, TrainerState, TrainerControl
# import numpy as np
# import torch
# import torch.nn.functional as F
# from google.colab import userdata

# batch_size = 2

# client = openai.OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# class SaveModelCallback(TrainerCallback):
#     def __init__(self, save_path, save_every=10000):
#         self.save_path = save_path
#         self.save_every = save_every

#     def on_step_end(self, args, state, control, model=None, **kwargs):
#         if state.global_step % self.save_every == 0 and state.global_step > 0:
#             if model is not None:
#                 print(f"[Step {state.global_step}] Saving model to {self.save_path}...")
#                 torch.save(model, self.save_path)


# class IterativeDiffusionEvalCallback(TrainerCallback):
#     def __init__(self, tokenizer, val_dataset, assistant_marker="Assistant:", max_iter=32):
#         self.tokenizer = tokenizer
#         self.dataset = val_dataset
#         self.max_iter = max_iter
#         self.assistant_marker_ids = tokenizer.encode(assistant_marker, add_special_tokens=False)
#         self.vocab_size = tokenizer.vocab_size
#         self.rng = np.random.default_rng()

#     def find_answer_start(self, input_ids):
#         for i in range(len(input_ids) - len(self.assistant_marker_ids) + 1):
#             if input_ids[i:i + len(self.assistant_marker_ids)] == self.assistant_marker_ids:
#                 return i + len(self.assistant_marker_ids)
#         return None

#     def noisify_answer(self, input_ids, answer_start):
#         noised = input_ids.copy()
#         answer_len = len(input_ids) - answer_start
#         indices = self.rng.choice(np.arange(answer_start, len(input_ids)), size=answer_len, replace=False)
#         for idx in indices:
#             noised[idx] = tokenizer.encode('MASK', add_special_tokens=False)[0]  # Overwrite with MASK or EOT
#         return noised

#     def generate_step(self, model, input_ids, answer_start):
#         input_tensor = torch.tensor([input_ids], dtype=torch.long).to(model.device)
#         with torch.no_grad():
#             logits = model(input_ids=input_tensor)["logits"]
#             probs = F.softmax(logits, dim=-1).squeeze()
#             probs = torch.clamp(probs, min=1e-8, max=1.0)
#             sampled = torch.multinomial(probs, num_samples=1).squeeze().tolist()
#         return input_ids[:answer_start] + sampled[answer_start:]

#     def decode(self, token_ids):
#         return self.tokenizer.decode(token_ids, skip_special_tokens=True).replace("\n", " ")

#     def on_evaluate(self, args, state: TrainerState, control: TrainerControl, model=None, **kwargs):
#         model.eval()
#         idx = int(self.rng.integers(0, len(self.dataset)))  # Cast to native Python int
#         example = self.dataset[idx]

#         input_ids = example["input_ids"]
#         label_ids = example["labels"]
#         answer_start = self.find_answer_start(input_ids)

#         if answer_start is None:
#             print("❌ 'Assistant:' not found in validation example.")
#             return

#         print("\n=== DIFFUSION VALIDATION SAMPLE ===")
#         print("Prompt         :", self.decode(input_ids[:answer_start]))
#         print("Ground Truth   :", self.decode(label_ids[answer_start:]))

#         # Start with fully noised answer
#         current = self.noisify_answer(input_ids.copy(), answer_start)
#         print("\nNoisy Init     :", self.decode(current[answer_start:]))

#         history = []
#         for i in range(self.max_iter):
#             current = self.generate_step(model, current, answer_start)
#             gen_text = self.decode(current[answer_start:])
#             history.append(current)

#             print(f"[{i:02d}] Step     :", gen_text)

#             if len(history) >= 3 and history[-1] == history[-2] == history[-3]:
#                 print(f"\n✅ Converged at step {i}")
#                 break

#         print("\nFinal Answer   :", self.decode(current[answer_start:]))

# # Pass callback to Trainer
# save_callback = SaveModelCallback("./drive/MyDrive/diffusion-model-llama-3.2-3B-finetuned-improvednoise_lora8percent_initnonoise_refined.pth")

# # Truncate or pad sequences to fixed length
# def pad_or_truncate(tensor, max_len, pad_value):
#     length = tensor.size(1)
#     if length > max_len:
#         return tensor[:, :max_len]
#     elif length < max_len:
#         pad_len = max_len - length
#         pad_tensor = torch.full((1, pad_len), pad_value, dtype=torch.long, device=tensor.device)
#         return torch.cat([tensor, pad_tensor], dim=1)
#     else:
#         return tensor

# def noisify_answer(input_ids, answer_start, threshold=1.0):
#     noised = input_ids.copy()
#     answer_len = len(input_ids) - answer_start
#     num_to_noise = int(threshold * answer_len)
#     if num_to_noise > 0:
#         indices = rng.choice(np.arange(answer_start, len(input_ids)), size=num_to_noise, replace=False)
#         for idx in indices:
#             noised[idx] = tokenizer.encode('MASK', add_special_tokens=False)[0]
#     return noised

# def get_noising_threshold(step, max_steps, sharpness=5.0):
#     x = step / max_steps
#     return (np.exp(-sharpness * x) - np.exp(-sharpness)) / (1 - np.exp(-sharpness))

# # === Diffusion-style one-step refinement ===
# def generate_diffusion_text(model, input_ids, answer_start):
#     model.eval()
#     with torch.no_grad():
#         input_tensor = torch.tensor([input_ids], dtype=torch.long).to(model.device)
#         logits = model(input_ids=input_tensor)["logits"]
#         probs = F.softmax(logits / 1.0, dim=-1).squeeze()
#         probs = torch.clamp(probs, min=1e-8, max=1.0)
#         sampled = torch.multinomial(probs, num_samples=1).squeeze().tolist()
#     return input_ids[:answer_start] + sampled[answer_start:]

# # === Utility: Locate assistant start in full input_ids ===
# def find_answer_start(input_ids, marker_ids):
#     for i in range(len(input_ids) - len(marker_ids) + 1):
#         if input_ids[i:i + len(marker_ids)] == marker_ids:
#             return i + len(marker_ids)
#     return None

# # === Custom Trainer ===
# class IterativeRefinementTrainer(Trainer):
#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         input_ids = inputs["input_ids"]
#         device = input_ids.device
#         B, T = input_ids.shape

#         # Setup
#         assistant_marker_ids = tokenizer.encode("Assistant:", add_special_tokens=False)
#         mask_token_id = tokenizer.encode("MASK", add_special_tokens=False)[0]
#         max_iterations = 64
#         input_size = T
#         rng = np.random.default_rng()

#         all_inputs, all_labels = [], []

#         for i in range(B):
#             input_ids_i = input_ids[i].tolist()
#             answer_start = find_answer_start(input_ids_i, assistant_marker_ids)
#             if answer_start is None:
#                 continue

#             prompt_tokens = input_ids_i[:answer_start]
#             prompt_str = tokenizer.decode(prompt_tokens, skip_special_tokens=True)

#             # Step 0: fully masked answer
#             current_tokens = prompt_tokens + [mask_token_id] * (T - answer_start)
#             history = []

#             # Iterative refinement loop
#             for step in range(max_iterations):
#                 # Step generation
#                 current_tokens = generate_diffusion_text(model, current_tokens, answer_start)
#                 # === Add noise to simulate diffusion (optional schedule) ===
#                 threshold = 0.5 * get_noising_threshold(step + 1, max_iterations)
#                 current_tokens = noisify_answer(current_tokens, answer_start, threshold=threshold)
#                 history.append(current_tokens)

#                 # Check for convergence (same output 2+ times)
#                 if len(history) >= 3 and history[-1] == history[-2] == history[-3]:
#                     print(f"✅ Converged at step {step}")
#                     break

#             # Final model output
#             final_output_str = tokenizer.decode(current_tokens, skip_special_tokens=True)

#             # Ask GPT for improvement
#             try:
#                 response = client.chat.completions.create(
#                     model="gpt-4.1-mini",
#                     messages=[
#                         {"role": "system", "content": "You are an expert editor improving AI-generated answers."},
#                         {"role": "user", "content": f"""Improve the answer in this instruction-following output,
#                           keeping the prompt unchanged. Try to keep the word order generally the same, making only SMALL adjustments.
#                           NO MAJOR CHANGES. Try to improve the answer by substitution or swapping of some words that are close together.
#                           Even if the answer is wrong, try to change it only slightly to improve it. Again, sentence structure should remain similar.
#                           If the answer is correct, you do not need to change anything!
#                           You only need to improve the answer, it does not need to be fully correct!
#                           It is really important that you try to use the words that are already in the answer as much as possible!!!.
#                           Respond ONLY with the part AFTER 'Assistant:', do NOT include the word Assistant itself.\n\n{final_output_str}"""}
#                     ],
#                     temperature=0.7,
#                 )
#                 improved_answer = response.choices[0].message.content.strip()

#                 print("Final model output:", tokenizer.decode(current_tokens[answer_start:], skip_special_tokens=True).replace("\n", " "))
#                 print("GPT-improved answer:", improved_answer.replace("\n", " "))

#                 input_str = final_output_str
#                 label_str = f"{prompt_str} {improved_answer}"

#                 input_ids_step = tokenizer(input_str, return_tensors="pt")["input_ids"].to(device)
#                 label_ids_step = tokenizer(label_str, return_tensors="pt")["input_ids"].to(device)

#                 all_inputs.append(pad_or_truncate(input_ids_step, input_size, tokenizer.eos_token_id))
#                 all_labels.append(pad_or_truncate(label_ids_step, input_size, -100))

#             except Exception as e:
#                 print(f"OpenAI error: {e}")
#                 continue

#         # Compute loss on final step only
#         if not all_inputs:
#             return torch.tensor(0.0, requires_grad=True).to(device)

#         input_batch = torch.cat(all_inputs, dim=0)
#         label_batch = torch.cat(all_labels, dim=0)

#         outputs = model(input_ids=input_batch, labels=label_batch)
#         return (outputs["loss"], outputs) if return_outputs else outputs["loss"]



# # 8. Set Up TrainingArguments
# training_args = TrainingArguments(
#     output_dir="./drive/MyDrive/diffusion-model-llama-3.2-3B-finetuned-improvednoise_lora8percent_initnonoise_refined.pth",
#     num_train_epochs=1,
#     gradient_accumulation_steps=1,
#     per_device_train_batch_size=batch_size,
#     per_device_eval_batch_size=batch_size,
#     eval_strategy="steps",
#     eval_steps=10,
#     save_strategy="steps",
#     save_total_limit=1,  # Keeps only the latest checkpoint
#     save_steps=10000,
#     logging_steps=10,
#     logging_dir="./logs",
#     report_to="wandb",
#     disable_tqdm=False,
#     weight_decay=0.01,
#     learning_rate=1e-6,
#     warmup_steps=10,
#     lr_scheduler_type="cosine",
#     remove_unused_columns=False,
#     fp16 = True,  # Disable FP16 when BF16 is used
#     dataloader_num_workers=4,  # Use multiple workers for faster data loading
# )

# val_subset = val_dataset.select(range(np.min([len(val_dataset),6])))

# trainer = IterativeRefinementTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_subset,
#     data_collator=data_collator,
#     callbacks=[IterativeDiffusionEvalCallback(tokenizer, val_dataset), save_callback]
# )

# trainer.train()


In [ ]:
# from transformers import Trainer

# class TwoPassDiffusionTrainer(Trainer):
#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         input_ids = inputs["input_ids"]
#         labels = inputs["labels"]
#         device = input_ids.device

#         # === First pass ===
#         outputs_1 = model(input_ids=input_ids, labels = labels)
#         logits_1 = outputs_1["logits"]
#         loss_1 = outputs_1["loss"]

#         with torch.no_grad():
#             probs_1 = F.softmax(logits_1 / 1.0, dim=-1)
#             probs_1 = torch.clamp(probs_1, min=1e-8, max=1.0)
#             B, T, V = probs_1.shape
#             sampled_1 = torch.multinomial(probs_1.view(-1, V), num_samples=1).squeeze(-1).view(B, T)


#         # === Second input: original first half + predicted second half ===
#         input_half = input_ids[:, :input_ids.shape[1] // 2]         # original instruction
#         predicted_half = sampled_1     # generated answer
#         second_input = torch.cat([input_half, predicted_half], dim=1)

#         # === Second pass ===
#         outputs_2 = model(input_ids=second_input, labels=labels)
#         loss_2 = outputs_2["loss"]

#         # === Combine losses ===
#         loss = loss_1 + loss_2

#         return (loss, outputs_2) if return_outputs else loss

# batch_size = 8

# # 8. Set Up TrainingArguments
# training_args = TrainingArguments(
#     output_dir="./drive/MyDrive/diffusion-model-llama-3.2-3GB-finetuned-randomnoised",
#     num_train_epochs=3,
#     gradient_accumulation_steps=1,
#     per_device_train_batch_size=batch_size,
#     per_device_eval_batch_size=batch_size,
#     eval_strategy="steps",
#     eval_steps=100,
#     save_strategy="steps",
#     save_total_limit=1,  # Keeps only the latest checkpoint
#     save_steps=10000,
#     logging_steps=50,
#     logging_dir="./logs",
#     report_to="wandb",
#     disable_tqdm=False,
#     weight_decay=0.01,
#     learning_rate=1e-5,
#     warmup_steps=1000,
#     lr_scheduler_type="cosine",
#     remove_unused_columns=False,
#     fp16 = True,  # Disable FP16 when BF16 is used
#     dataloader_num_workers=4,  # Use multiple workers for faster data loading
# )

# trainer = TwoPassDiffusionTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_subset,
#     data_collator=data_collator,
#     callbacks=[diffusion_callback, save_callback]
# )

# trainer.train()

In [ ]:
# # torch.save(model.state_dict(), "diffusion_model.pth")
torch.save(model, "./drive/MyDrive/diffusion-model-llama-3.2-3B-finetuned-improvednoise_lora8percent_initnonoise.pth")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import time
import torch
import numpy as np

# === Set up widgets ===
input_box = widgets.Text(
    value='What do you know about the city of New York?',
    placeholder='Type your question here',
    description='User:',
    layout=widgets.Layout(width='80%')
)
eot_slider = widgets.FloatSlider(
    value=0.4,
    min=0.0,
    max=1.0,
    step=0.05,
    description='↓ = longer answers:',
    readout_format='.2f',
    continuous_update=False,
    style={'description_width': '180px'},
    layout=widgets.Layout(width='80%')
)

iteration_slider = widgets.IntSlider(
    value=64,
    min=1,
    max=128,
    step=1,
    description='↑ = more iterations:',
    continuous_update=False,
    style={'description_width': '180px'},
    layout=widgets.Layout(width='80%')
)

sharpness_slider = widgets.FloatSlider(
    value=5.0,
    min=1.0,
    max=20.0,
    step=0.5,
    description='↓ = more noising:',
    readout_format='.1f',
    continuous_update=False,
    style={'description_width': '180px'},
    layout=widgets.Layout(width='80%')
)

submit_button = widgets.Button(
    description='Submit',
    button_style='primary',
    tooltip='Click to generate answer'
)

output_area = widgets.Output()

# === Noising function with EOT probability scaling ===
def noisify_answer(input_ids, answer_start, threshold=1.0, eot_weight=1.0):
    noised = input_ids.copy()
    answer_len = len(input_ids) - answer_start
    num_to_noise = int(threshold * answer_len)
    if num_to_noise > 0:
        indices = rng.choice(np.arange(answer_start, len(input_ids)), size=num_to_noise, replace=False)

        # Interpolate token probabilities with and without EOT
        mixed_probs = token_probabilities.copy()
        mixed_probs[eot_token_id] *= eot_weight
        mixed_probs /= mixed_probs.sum()

        noise = rng.choice(np.arange(vocab_size), size=num_to_noise, p=mixed_probs)
        for idx, val in zip(indices, noise):
            noised[idx] = val
    return noised

# === Main interaction logic ===
def on_submit_clicked(b):
    question = input_box.value.strip()
    eot_weight = eot_slider.value
    max_it = iteration_slider.value
    sharpness = sharpness_slider.value
    with output_area:
        clear_output(wait=True)
        display(Markdown(f"### User: {question}"))

        prompt = f"User: {question}\nAssistant:"
        input_ids = tokenizer.encode(prompt, add_special_tokens=False)
        answer_start = find_answer_start(input_ids, assistant_marker_ids)
        if answer_start is None:
            display(Markdown("**Error:** Could not find Assistant marker in input."))
            return

        pad_token = tokenizer.pad_token_id or tokenizer.eos_token_id
        if len(input_ids) < 256:
            input_ids += [pad_token] * (256 - len(input_ids))
        else:
            input_ids = input_ids[:256]

        ori_input_tokens = input_ids
        current_tokens = noisify_answer(ori_input_tokens, answer_start, threshold=1.0, eot_weight=eot_weight)

        last_tokens = []

        for i in range(max_it):
            generated_tokens = generate_diffusion_text(model, current_tokens, answer_start)
            current_tokens = generated_tokens

            clear_output(wait=True)
            display(Markdown(f"### User: {question}"))
            display(Markdown(f"### Iteration {i+1}/{max_it}"))
            display(Markdown("**Diffusion Output:**<br>" + decode_tokens_safe(generated_tokens[answer_start:]).replace("\n", "<br>")))

            last_tokens.append(generated_tokens)
            if len(last_tokens) > 3:
                last_tokens.pop(0)
            if len(last_tokens) == 3 and last_tokens[0] == last_tokens[1] == last_tokens[2]:
                break

            threshold = get_noising_schedule(i, max_it, sharpness=sharpness)
            current_tokens = noisify_answer(generated_tokens, answer_start, threshold=threshold, eot_weight=eot_weight)
            time.sleep(0.01)

        display(Markdown("**Final Output:** " + decode_tokens_safe(current_tokens[answer_start:])))

submit_button.on_click(on_submit_clicked)

# === Display the interface ===
display(widgets.VBox([input_box, eot_slider, iteration_slider, sharpness_slider, submit_button, output_area]))

In [ ]:
import gradio as gr
import numpy as np
import time

placeholder = "What do you know about the city of New York?"

# === Noising function with EOT probability scaling ===
def noisify_answer(input_ids, answer_start, threshold=1.0, eot_weight=1.0):
    noised = input_ids.copy()
    answer_len = len(input_ids) - answer_start
    num_to_noise = int(threshold * answer_len)
    if num_to_noise > 0:
        indices = rng.choice(np.arange(answer_start, len(input_ids)), size=num_to_noise, replace=False)

        mixed_probs = token_probabilities.copy()
        mixed_probs[eot_token_id] *= eot_weight
        mixed_probs /= mixed_probs.sum()

        noise = rng.choice(np.arange(vocab_size), size=num_to_noise, p=mixed_probs)
        for idx, val in zip(indices, noise):
            noised[idx] = val
    return noised

# === Gradio streaming generator ===
def diffusion_chat(question, eot_weight, max_it, sharpness):
    if question.strip() == "":
        question = placeholder

    prompt = f"User: {question}\nAssistant:"
    input_ids = tokenizer.encode(prompt, add_special_tokens=False)
    answer_start = find_answer_start(input_ids, assistant_marker_ids)
    if answer_start is None:
        yield "Error: Could not find Assistant marker in input."
        return

    pad_token = tokenizer.pad_token_id or tokenizer.eos_token_id
    if len(input_ids) < 256:
        input_ids += [pad_token] * (256 - len(input_ids))
    else:
        input_ids = input_ids[:256]

    ori_input_tokens = input_ids
    current_tokens = noisify_answer(ori_input_tokens, answer_start, threshold=1.0, eot_weight=eot_weight)

    last_tokens = []
    prev_decoded_tokens = []

    for i in range(max_it):
        generated_tokens = generate_diffusion_text(model, current_tokens, answer_start)
        current_tokens = generated_tokens

        decoded_ids = current_tokens[answer_start:]
        decoded_tokens = tokenizer.convert_ids_to_tokens(decoded_ids)

        # Filter out EOT tokens
        filtered_tokens = [tok for tok in decoded_tokens if tokenizer.convert_tokens_to_ids(tok) != eot_token_id]
        filtered_prev_tokens = [tok for tok in prev_decoded_tokens if tokenizer.convert_tokens_to_ids(tok) != eot_token_id] if prev_decoded_tokens else []

        # Highlight differences in red
        if filtered_prev_tokens:
            highlighted = []
            for tok_new, tok_old in zip(filtered_tokens, filtered_prev_tokens):
                if tok_new != tok_old:
                    highlighted.append(f'<span style="color:green">{tokenizer.convert_tokens_to_string([tok_new])}</span>')
                else:
                    highlighted.append(tokenizer.convert_tokens_to_string([tok_new]))
        else:
            highlighted = [tokenizer.convert_tokens_to_string([tok]) for tok in filtered_tokens]

        prev_decoded_tokens = decoded_tokens

        yield f"<b>Iteration {i+1}/{max_it} (running):</b><br>" + "".join(highlighted)

        last_tokens.append(generated_tokens)
        if len(last_tokens) > 3:
            last_tokens.pop(0)
        if len(last_tokens) == 3 and last_tokens[0] == last_tokens[1] == last_tokens[2]:
            yield f"<b>Stopped early after {i+1} iterations.</b>"
            break

        threshold = get_noising_schedule(i, max_it, sharpness=sharpness)
        current_tokens = noisify_answer(generated_tokens, answer_start, threshold=threshold, eot_weight=eot_weight)
        time.sleep(0.01)

    final_tokens = tokenizer.convert_ids_to_tokens(current_tokens[answer_start:])
    final_tokens = [tok for tok in final_tokens if tokenizer.convert_tokens_to_ids(tok) != eot_token_id]
    final_output = tokenizer.convert_tokens_to_string(final_tokens)
    yield f"<b>Final Output (after {i+1} iterations):</b><br>" + final_output

# === Launch Gradio app ===
demo = gr.Interface(
    fn=diffusion_chat,
    inputs=[
        gr.Textbox(label="User Question", lines=2, placeholder=placeholder),
        gr.Slider(0, 1, value=0.4, step=0.05, label="↓ = longer answers (EOT weight)"),
        gr.Slider(1, 512, value=64, step=1, label="↑ = more iterations"),
        gr.Slider(1.0, 20.0, value=5.0, step=0.5, label="↓ = more noising (sharpness)")
    ],
    outputs=gr.HTML(label="Diffusion Output"),
    title="Diffusion Language Model Chat",
    description="This interface runs a diffusion-based language model to generate answers progressively."
)

demo.launch(share=True)

In [ ]:
# from types import MethodType

# for name, module in model.named_modules():
#     if "BidirectionalLlamaAttention" in str(type(module)):
#         print(f"Found custom attention layer in: {name}")

# for name, module in model.named_modules():
#     if isinstance(module, BidirectionalLlamaAttention):
#         print(f"{name}: masking = {module.masking}")

# for name, module in model.named_modules():
#     if isinstance(module, BidirectionalLlamaAttention):
#         module.masking = 'bidirectional'  # Update to new masking mode

# for name, module in model.named_modules():
#     if isinstance(module, BidirectionalLlamaAttention):
#         print(f"{name}: masking = {module.masking}")

In [ ]:

# import torch
# import numpy as np
# from torch.utils.data import DataLoader
# from transformers import AdamW
# import gc
# import openai

# torch.cuda.empty_cache()
# gc.collect()

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# batch_size = 4

# # Define dataloaders
# train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
# val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=data_collator)

# # Define optimizer
# optimizer = AdamW(model.parameters(), lr=5e-6, weight_decay=0.01)

# # Set model to training mode
# model.train()
# model.to(device)

# num_epochs = 1

# for epoch in range(num_epochs):
#     for i, batch in enumerate(train_dataloader):
#         input_ids = batch["input_ids"].to(device)
#         labels = batch["labels"].to(device)

#         optimizer.zero_grad()  # Reset gradients before accumulation

#         # ====== First Pass ======
#         outputs = model(input_ids=input_ids)
#         logits = outputs['logits']
#         loss1 = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))
#         loss1.backward()  # Accumulate gradients

#         # ====== Generate Predictions ======
#         with torch.no_grad():
#             predicted_tokens = torch.argmax(logits, dim=-1)  # Greedy decoding
#             input_ids[:, -input_size // 2:] = predicted_tokens  # Replace last half with generated tokens

#         # ====== Second Pass (Train Again on Predictions) ======
#         outputs = model(input_ids=input_ids)
#         logits = outputs['logits']
#         loss2 = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))
#         loss2.backward()  # Accumulate gradients

#         # ====== Perform Optimization Step After Both Passes ======
#         optimizer.step()

#         # ====== Evaluate on Validation Set Every 50 Steps ======
#         if i % 50 == 0:
#             model.eval()  # Set model to evaluation mode
#             val_loss = 0
#             with torch.no_grad():
#                 for val_batch in val_dataloader:
#                     val_input_ids = val_batch["input_ids"].to(device)
#                     val_labels = val_batch["labels"].to(device)

#                     val_outputs = model(input_ids=val_input_ids)
#                     val_logits = val_outputs['logits']
#                     val_loss += torch.nn.functional.cross_entropy(val_logits.view(-1, val_logits.size(-1)), val_labels.view(-1)).item()

#             avg_val_loss = val_loss / len(val_dataloader)
#             print(f"Epoch {epoch + 1}/{num_epochs}, Step {i}, Validation Loss: {avg_val_loss}")

#             model.train()  # Restore training mode

#     print(f"Epoch {epoch + 1}/{num_epochs} completed.")

# print("Training completed.")


In [ ]:

# def generate_text_top_k(prompt, max_new_tokens=100, k=3):
#     input_ids = torch.tensor([encode_text(prompt)], dtype=torch.long).to(model.device)
#     model.eval()
#     with torch.no_grad():
#         for _ in range(max_new_tokens):
#             outputs = model(input_ids=input_ids)
#             logits = outputs.logits
#             next_token_logits = logits[0, -1, :]

#             # Keep only the top-k highest probability tokens
#             top_k_logits, top_k_indices = torch.topk(next_token_logits, k)
#             probabilities = torch.nn.functional.softmax(top_k_logits, dim=-1)
#             next_token_id = torch.multinomial(probabilities, num_samples=1)

#             # Map back to original indices
#             next_token_id = top_k_indices[next_token_id].unsqueeze(0)
#             input_ids = torch.cat([input_ids, next_token_id], dim=1)

#     return decode_tokens(input_ids[0].tolist())

# print(generate_text_top_k("<bos> 16 x 6 = ", max_new_tokens=200, k=3))


In [ ]:
    # def custom_attn(self, query, key, value, attention_mask=None, head_mask=None):
    #     """ 🚀 Custom attention function replacing `_attn()` """

    #     attn_weights = torch.matmul(query, key.transpose(-1, -2))

    #     if self.scale_attn_weights:
    #         attn_weights = attn_weights / (float(value.size(-1)) ** 0.5)

    #     # Dynamically enforce unidirectional attention if enabled
    #     if self.force_unidirectional:
    #         seq_len = attn_weights.size(-1)
    #         causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=attn_weights.device)).unsqueeze(0).unsqueeze(0)
    #         attn_weights = attn_weights.masked_fill(causal_mask == 0, float('-inf'))

    #     # Apply padding mask if provided
    #     if attention_mask is not None:
    #         attn_weights = attn_weights + attention_mask

    #     attn_weights = nn.Softmax(dim=-1)(attn_weights)
    #     attn_weights = self.attn_dropout(attn_weights)

    #     if head_mask is not None:
    #         attn_weights = attn_weights * head_mask

    #     attn_output = torch.matmul(attn_weights, value)

    #     return attn_output, attn_weights